# Stacks


## Topic overview

LIFO structure for balanced matching and monotonic patterns.

## Pattern-recognition rules

- Balanced-brackets validation.
- Monotonic stack for next-greater / next-smaller.
- Expression parsing.

## Common data structures

- `list` used as a stack
- `collections.deque`

## Standard complexity expectations

- Monotonic-stack problems are typically O(n).

## Common mistakes

- Popping from an empty stack.
- Confusing when to pop vs. push.

## Original illustrative example

In [ ]:
# Replace with an ORIGINAL example. Do not paste external
# problem statements. See src/algorithms/ for reusable helpers.
example_input = []
example_expected = None

## Add solved problems below

Each new sub-section should follow the template in `../templates/notebook_template.ipynb`.

---

# Min Stack

## Metadata

- Source: NeetCode
- Problem URL: https://neetcode.io/problems/minimum-stack
- Difficulty: Medium
- Topic: Stacks
- Solution path: `../Data Structures & Algorithms/min-stack/submission-0.py`


## 1. Setup

We need a stack supporting four operations:

- `push(val)`: add `val` to the top.
- `pop()`: remove the top value.
- `top()`: return the top value.
- `getMin()`: return the minimum value currently stored.

Every operation must run in $O(1)$ time.

A normal stack handles `push`, `pop`, and `top` in $O(1)$, but finding the
minimum by scanning the whole stack would take $O(n)$. So we need to store
additional information.


## 2. Governing principle

Use two synchronized stacks:

- `stack`: stores every value.
- `min_stack`: stores the minimum value corresponding to each stack state.

When a value is pushed, we also push the new minimum onto `min_stack`.
Therefore both stacks always have the same length, and

$$\texttt{min\_stack[-1]}$$

is always the minimum of all values currently in `stack`.


## 3. Algorithm design

**`push(val)`** — push `val` onto the main stack. For the minimum stack:
if this is the first value, then `val` is automatically the minimum;
otherwise compare `val` with the previous minimum,

$$\text{new minimum} = \min(\text{val},\ \text{current minimum})$$

and push that result onto `min_stack`.

**`pop()`** — remove the top element from both stacks, because both entries
represent the same stack state.

**`top()`** — return the last element of the main stack.

**`getMin()`** — return the last element of the minimum stack.


## 4. Pseudocode

```text
class MinStack:
    initialize:
        stack = empty list
        min_stack = empty list

    push(val):
        append val to stack
        if min_stack is empty:
            append val to min_stack
        else:
            new_minimum = minimum(val, top of min_stack)
            append new_minimum to min_stack

    pop():
        remove top of stack
        remove top of min_stack

    top():
        return top of stack

    getMin():
        return top of min_stack
```


## 5. Python solution

In [ ]:
class MinStack:
    def __init__(self) -> None:
        """Initialize the main stack and its synchronized minimum stack."""
        self.stack: list[int] = []
        self.min_stack: list[int] = []

    def push(self, val: int) -> None:
        """Push val and record the minimum for the resulting stack state."""
        self.stack.append(val)
        if not self.min_stack:
            # The first value is automatically the current minimum.
            self.min_stack.append(val)
        else:
            # Preserve the smaller of val and the previous minimum.
            current_minimum = self.min_stack[-1]
            self.min_stack.append(min(val, current_minimum))

    def pop(self) -> None:
        """Remove the top value and its corresponding minimum state."""
        self.stack.pop()
        self.min_stack.pop()

    def top(self) -> int:
        """Return the top value without removing it."""
        return self.stack[-1]

    def getMin(self) -> int:
        """Return the minimum value currently in the stack."""
        return self.min_stack[-1]

## 6. Trace the example

Start with `stack = []` and `min_stack = []`.

| Operation | `stack` | `min_stack` | Returns | Why |
|---|---|---|---|---|
| `push(1)` | `[1]` | `[1]` | | Stack was empty, so `1` is the minimum |
| `push(2)` | `[1, 2]` | `[1, 1]` | | $\min(2, 1) = 1$ |
| `push(0)` | `[1, 2, 0]` | `[1, 1, 0]` | | $\min(0, 1) = 0$ |
| `getMin()` | `[1, 2, 0]` | `[1, 1, 0]` | `0` | `min_stack[-1]` |
| `pop()` | `[1, 2]` | `[1, 1]` | | Both stacks pop together |
| `top()` | `[1, 2]` | `[1, 1]` | `2` | `stack[-1]` |
| `getMin()` | `[1, 2]` | `[1, 1]` | `1` | `min_stack[-1]`, recovered for free |

The last line is the whole point: popping the minimum `0` did not destroy the
knowledge that `1` was the minimum beforehand, because that fact was stored at
its own depth.


In [ ]:
min_stack = MinStack()
min_stack.push(1)
min_stack.push(2)
min_stack.push(0)

assert min_stack.getMin() == 0
min_stack.pop()
assert min_stack.top() == 2
assert min_stack.getMin() == 1

print("Example passed.")

## 7. Why duplicate minimums are necessary

Suppose the input is `push(2)`, `push(1)`, `push(1)`. The stacks become:

```text
stack     = [2, 1, 1]
min_stack = [2, 1, 1]
```

After one `pop()`:

```text
stack     = [2, 1]
min_stack = [2, 1]
```

The minimum is still `1`, which is correct. Storing a minimum for *every*
stack state avoids special handling for duplicate minimum values. An
optimization that only records a new minimum when `val < min_stack[-1]`
would pop the stored `1` here and wrongly report `2`.


In [ ]:
# Duplicate-minimum check: popping one copy of the minimum must keep the other.
duplicates = MinStack()
for value in (2, 1, 1):
    duplicates.push(value)

duplicates.pop()
assert duplicates.getMin() == 1, "duplicate minimum was lost"

print("Duplicate minimums handled correctly.")

## 8. Complexity analysis

Each operation performs only a constant number of list operations:

| Operation | Time |
|---|---|
| `push` | $O(1)$ |
| `pop` | $O(1)$ |
| `top` | $O(1)$ |
| `getMin` | $O(1)$ |

For $n$ stored elements, both stacks can contain $n$ values, so

$$\text{Space complexity} = O(n)$$


## 9. Alternative: one stack of pairs

Each stack element can instead store `(value, minimum_at_this_point)`.


In [ ]:
class MinStackPairs:
    def __init__(self) -> None:
        self.stack: list[tuple[int, int]] = []

    def push(self, val: int) -> None:
        if not self.stack:
            current_minimum = val
        else:
            previous_minimum = self.stack[-1][1]
            current_minimum = min(val, previous_minimum)
        self.stack.append((val, current_minimum))

    def pop(self) -> None:
        self.stack.pop()

    def top(self) -> int:
        return self.stack[-1][0]

    def getMin(self) -> int:
        return self.stack[-1][1]

Both methods have the same complexity. The two-stack version is usually
easier to understand initially, while the pair-stack version keeps all state
in one data structure.

**Recommended solution:** synchronized main stack and minimum stack.


## Randomized cross-check

Confirm both implementations against a brute-force `min()` over a plain list.


In [ ]:
import random


def cross_check(stack_class, operations: int = 20_000, seed: int = 0) -> None:
    """Compare stack_class against a plain list using min() as ground truth."""
    random.seed(seed)
    subject = stack_class()
    reference: list[int] = []

    for _ in range(operations):
        if reference and random.random() < 0.4:
            subject.pop()
            reference.pop()
        else:
            value = random.randint(-10, 10)
            subject.push(value)
            reference.append(value)

        if reference:
            assert subject.top() == reference[-1]
            assert subject.getMin() == min(reference)


cross_check(MinStack)
cross_check(MinStackPairs)
print("Both implementations passed 20,000 randomized operations.")

## Pattern recognition

The reusable idea is **carrying an auxiliary value alongside each stack
frame**. Whenever a query must be answered in $O(1)$ but depends on the whole
current contents, ask whether the answer can be computed at push time and
stored at that depth. The same trick answers "max in stack" or "sum of stack"
by swapping `min` for `max` or `+`.

Related: this is *not* a monotonic stack. A monotonic stack discards elements
to keep an ordering invariant; here nothing is discarded, and `min_stack` is
merely non-increasing as a consequence of how it is built.

## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| | | |
